# Using AI to Secure AI: RL for Attack Path Simulation
## MMAI 845 -- Reinforcement Learning | Syed Ali Turab

This notebook presents the complete experimental analysis for the RL attack-path
simulation project. It walks through the environment design, trains PPO and DQN
agents under baseline and stealth reward regimes, evaluates their performance,
and extracts security-relevant findings.

**Structure:**
1. Environment Walkthrough -- topology, observation/action spaces, MDP formulation
2. Network Topology Diagram -- visual map of the enterprise AI infrastructure
3. MITRE ATT&CK Mapping -- bridging NASim actions to real-world attack techniques
4. Training Results -- learning curves for PPO and DQN (baseline vs stealth)
5. Evaluation Comparison -- head-to-head metrics table
6. Attack Path Analysis -- action-to-host mapping, pivot point identification, heatmap
7. Reward Decomposition -- stealth trade-off visualisation
8. What-If Topology Analysis -- counterfactual firewall changes and their impact
9. Detection Sensitivity Sweep -- success rate vs detection threshold
10. Automated Pentest Report -- generated security assessment document
11. Key Findings -- answers to the three research questions

---
## 0. Setup and Imports

In [ ]:
import sys
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Project imports
sys.path.insert(0, str(Path.cwd().parent))

from environments.network_config import make_env, AI_INFRA_HOSTS, _NETWORK_YAML
from environments.stealth_wrapper import make_stealth_env, StealthAwareWrapper
from agents.ppo_agent import PPOAttackAgent
from agents.dqn_agent import DQNAttackAgent
from analysis.attack_path import build_action_map, interpret_path, summarise_path, find_common_pivots
from analysis.mitre_mapping import generate_mitre_summary_table, map_path_to_mitre, MITRE_MAPPING
from analysis.what_if import MODIFICATIONS, evaluate_with_modified_topology, run_all_what_if
from analysis.topology_diagram import generate_topology_diagram
from analysis.report_generator import generate_pentest_report
from training.evaluate import evaluate_agent

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

RESULTS_DIR = Path('../results')
PLOTS_DIR = RESULTS_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

---
## 1. Environment Walkthrough

The environment is built on NASim (Network Attack Simulator), a Gymnasium-compatible
discrete MDP. The agent represents an attacker starting at the internet boundary
who must pivot through firewalled subnets to reach high-value AI infrastructure.

### MDP Formulation

| Component | Description |
|---|---|
| **State** | Flat vector encoding per-host discovery status, access level, and compromise status |
| **Actions** | Discrete set of (host, action_type) pairs: subnet scan, OS scan, service scan, exploit, privilege escalation |
| **Transitions** | Stochastic: each exploit has a success probability (0.7--0.9) |
| **Rewards** | +value on first compromise of each host; penalty for invalid actions |
| **Episode termination** | All sensitive hosts compromised, or detection threshold exceeded (stealth mode) |

In [ ]:
# Create the environment and inspect its structure
env = make_env()

print('=== Environment Summary ===')
print(f'Observation space: {env.observation_space}')
print(f'Observation shape:  {env.observation_space.shape}')
print(f'Observation dtype:  {env.observation_space.dtype}')
print(f'Action space:       {env.action_space}')
print(f'Number of actions:  {env.action_space.n}')
print()
print('Both PPO (continuous policy over discrete actions) and DQN (Q-value per')
print('discrete action) are compatible with this space. The flat observation')
print('vector works with MLP policies without any preprocessing.')
print()

# Reset and show initial observation
obs, info = env.reset()
print(f'Initial observation (first 20 elements): {obs[:20]}')
print(f'Observation vector length: {len(obs)}')
print(f'Info keys: {list(info.keys())}')

In [ ]:
# Demonstrate stepping through the environment
print('=== Stepping Through the Environment ===')
print()
env.reset()
for step in range(5):
    action = int(env.action_space.sample())
    obs, reward, terminated, truncated, info = env.step(action)
    print(f'Step {step+1}: action={action:3d} | reward={reward:7.2f} | '
          f'terminated={terminated} | truncated={truncated}')
print()
print('Negative rewards indicate invalid actions (preconditions not met).')
print('Positive rewards indicate successful first-time host compromise.')

In [ ]:
# Show the sensitive hosts (AI infrastructure targets)
print('=== AI Infrastructure Targets ===')
print()
targets = {
    (3, 0): ('LLM API Server', 200),
    (3, 1): ('Vector Database', 200),
    (3, 2): ('Model Repository', 150),
    (4, 0): ('Training Data Lake', 300),
}
print(f'{"Host":>20} {"Subnet":>10} {"Value":>8}')
print('-' * 42)
for (subnet, host), (name, value) in targets.items():
    print(f'{name:>20} {subnet:>10} {value:>8}')
print()
total_value = sum(v for _, v in targets.values())
print(f'Total sensitive host value: {total_value}')
print(f'An agent that compromises all AI assets earns {total_value} reward points.')

---
## 2. Network Topology Diagram

In [ ]:
# Generate the topology diagram
diagram_path = str(PLOTS_DIR / 'network_topology.png')
generate_topology_diagram(output_path=diagram_path)

# Display it
from IPython.display import Image, display
display(Image(filename=diagram_path, width=900))

The diagram shows the 5-subnet enterprise network. Key observations:

- The attacker must traverse **at least 3 subnets** to reach AI infrastructure
- Firewall rules restrict which hosts can communicate across subnet boundaries
- The LLM API Server (3,0) is the critical gateway -- it accepts HTTP from the
  Dev Server and SSH from Internal Services, making it the primary pivot point
- The Data Lake (4,0) has the highest value (300) and is accessible only via
  SSH from the LLM API Server or Model Repository

---
## 3. MITRE ATT&CK Mapping

Each NASim action type maps to one or more techniques in the MITRE ATT&CK
framework, the industry standard for describing adversary behaviour. This
mapping enables security teams to interpret the RL agent's learned strategy
in terms they use for threat modelling and incident response.

In [ ]:
mitre_table = generate_mitre_summary_table()
mitre_df = pd.DataFrame(mitre_table)
mitre_df = mitre_df[mitre_df['tactic'] != 'N/A']  # Exclude noop

print('=== NASim Actions to MITRE ATT&CK Mapping ===')
print()
display_df = mitre_df[['nasim_action', 'tactic', 'technique_id', 'technique_name']].copy()
display_df.columns = ['NASim Action', 'ATT&CK Tactic', 'Technique ID', 'Technique Name']
display(display_df.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

The agent's learned attack chain follows the standard kill chain:

1. **Discovery** (T1046, T1082) -- scan subnets and fingerprint hosts
2. **Initial Access** (T1190) -- exploit public-facing web applications in the DMZ
3. **Lateral Movement** (T1021) -- pivot via SSH, SMB, or RDP to internal hosts
4. **Privilege Escalation** (T1068) -- escalate from user to root on target hosts

This mapping is not cosmetic. It means the simulation's outputs can be directly
compared against real threat intelligence reports and used to validate detection
rules written in ATT&CK-based frameworks (e.g. Sigma, YARA).

---
## 4. Training Results

Training is performed via the Docker pipeline or the CLI:
```bash
docker compose run train            # PPO + DQN baseline
docker compose run train-stealth    # PPO + DQN stealth
```

The cells below load saved results from `results/`. If training has not been
run yet, placeholder messages will appear.

In [ ]:
# Load training metadata
conditions = ['ppo_baseline', 'ppo_stealth', 'dqn_baseline', 'dqn_stealth']
train_meta = {}

for cond in conditions:
    meta_path = RESULTS_DIR / cond / 'train_meta.json'
    if meta_path.exists():
        with open(meta_path) as f:
            train_meta[cond] = json.load(f)
        print(f'[OK] Loaded {cond}: {train_meta[cond].get("total_timesteps", "?")} steps, '
              f'{train_meta[cond].get("wall_time_seconds", "?")}s')
    else:
        print(f'[--] {cond}: not found. Run training first.')

if not train_meta:
    print()
    print('No training results found. Run:')
    print('  docker compose run train')
    print('  docker compose run train-stealth')

In [ ]:
# Plot training curves from Monitor CSV logs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'ppo_baseline': '#2196F3', 'dqn_baseline': '#FF5722',
          'ppo_stealth': '#7B1FA2', 'dqn_stealth': '#E65100'}

found_data = False
for cond in conditions:
    monitor_files = list((RESULTS_DIR / cond).rglob('monitor.csv')) if (RESULTS_DIR / cond).exists() else []
    for mf in monitor_files:
        try:
            df = pd.read_csv(mf, skiprows=1)
            if 'r' in df.columns:
                smoothed_r = df['r'].rolling(50, min_periods=1).mean()
                axes[0].plot(smoothed_r, label=cond, color=colors.get(cond, '#999'),
                            linewidth=1.5, alpha=0.85)
                found_data = True
            if 'l' in df.columns:
                smoothed_l = df['l'].rolling(50, min_periods=1).mean()
                axes[1].plot(smoothed_l, label=cond, color=colors.get(cond, '#999'),
                            linewidth=1.5, alpha=0.85)
        except Exception:
            pass

if found_data:
    axes[0].set_title('Episode Reward (smoothed)', fontweight='bold')
    axes[0].set_xlabel('Episode')
    axes[0].set_ylabel('Reward')
    axes[0].legend(fontsize=8)
    axes[1].set_title('Episode Length (smoothed)', fontweight='bold')
    axes[1].set_xlabel('Episode')
    axes[1].set_ylabel('Steps')
    axes[1].legend(fontsize=8)
    plt.suptitle('Training Curves: PPO vs DQN (Baseline and Stealth)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    plt.close()
    print('No monitor.csv files found. Training curves will appear after running training.')

---
## 5. Evaluation Comparison

After training, agents are evaluated over 100 episodes each. The key metrics:

| Metric | Definition |
|---|---|
| **Success Rate** | Fraction of episodes where total reward >= 100 (agent reached AI infrastructure) |
| **Catch Rate** | Fraction of episodes terminated by detection (stealth mode only) |
| **Mean Reward** | Average episode return |
| **Mean Steps** | Average episode length |

In [ ]:
# Load evaluation results
eval_path = RESULTS_DIR / 'eval_results.json'
if eval_path.exists():
    with open(eval_path) as f:
        eval_results = json.load(f)

    rows = []
    for agent_name, metrics in eval_results.items():
        rows.append({
            'Agent': agent_name.upper(),
            'Mean Reward': f"{metrics.get('mean_reward', 0):.2f}",
            'Std Reward': f"{metrics.get('std_reward', 0):.2f}",
            'Success Rate': f"{metrics.get('success_rate', 0):.1%}",
            'Catch Rate': f"{metrics.get('catch_rate', 0):.1%}",
            'Mean Steps': f"{metrics.get('mean_steps', 0):.1f}",
        })

    eval_df = pd.DataFrame(rows)
    print('=== Evaluation Results (100 episodes per agent) ===')
    display(eval_df.style.hide(axis='index').set_properties(**{'text-align': 'center'}))
else:
    print('No eval_results.json found. Run:')
    print('  docker compose run evaluate')

In [ ]:
# Comparison bar chart
if eval_path.exists():
    with open(eval_path) as f:
        eval_data = json.load(f)

    metrics_to_plot = ['mean_reward', 'mean_steps', 'success_rate', 'catch_rate']
    metric_labels = ['Mean Reward', 'Mean Steps', 'Success Rate', 'Catch Rate']
    agents = list(eval_data.keys())
    x = np.arange(len(metrics_to_plot))
    width = 0.35
    bar_colors = ['#2196F3', '#FF5722', '#7B1FA2', '#E65100']

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, agent in enumerate(agents):
        vals = [eval_data[agent].get(m, 0) or 0 for m in metrics_to_plot]
        offset = (i - len(agents)/2) * width + width/2
        ax.bar(x + offset, vals, width*0.9, label=agent.upper(),
               color=bar_colors[i % len(bar_colors)])

    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels)
    ax.set_title('PPO vs DQN: Evaluation Metrics', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'comparison_bar.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 6. Attack Path Analysis

This section maps raw action indices back to human-readable host names and
action types, answering the core security question: **which hosts serve as
stepping stones to AI infrastructure?**

In [ ]:
# Build action map from the environment
env = make_env()
action_map = build_action_map(env)
print(f'Total actions in environment: {len(action_map)}')
print()
print('=== Sample Action Mappings ===')
print(f'{"Index":>6} {"Type":>20} {"Name":>20} {"Target Host":>20}')
print('-' * 70)
for entry in action_map[:15]:
    print(f'{entry["action_idx"]:>6} {entry["action_type"]:>20} '
          f'{entry["action_name"]:>20} {entry["host_name"]:>20}')

In [ ]:
# Run a single episode with a trained agent (or random if no model saved)
# and interpret the attack path
ppo_model_path = RESULTS_DIR / 'ppo_baseline' / 'final_model'
eval_env = make_env()

if ppo_model_path.exists() or (ppo_model_path.parent / 'final_model.zip').exists():
    agent = PPOAttackAgent.load(str(ppo_model_path), eval_env)
    print('Loaded trained PPO agent.')
else:
    agent = PPOAttackAgent(env=eval_env, tensorboard_log=None)
    print('No trained model found -- using untrained agent for demonstration.')

result = agent.run_episode(eval_env, deterministic=True)
print(f'Episode reward: {result["total_reward"]:.2f}')
print(f'Episode steps:  {result["steps"]}')
print()

# Interpret the path
interpreted = interpret_path(result['path'], action_map)
summary = summarise_path(interpreted)

print('=== Attack Path Summary ===')
print(f'Total steps:     {summary["total_steps"]}')
print(f'Action types:    {summary["action_type_counts"]}')
print(f'Pivot chain:     {" -> ".join(summary["pivot_chain"])}')
print()

# Show first 20 steps of the interpreted path
print('=== First 20 Steps ===')
print(f'{"Step":>5} {"Action Type":>20} {"Target Host":>20} {"Subnet":>20}')
print('-' * 68)
for step in interpreted[:20]:
    print(f'{step["step"]:>5} {step["action_type"]:>20} '
          f'{step["host_name"]:>20} {step.get("subnet_name", "?"):>20}')

In [ ]:
# Attack path heatmap: which connections does the agent use most?
# Run multiple episodes and aggregate
n_heatmap_episodes = 50
all_paths = []
for _ in range(n_heatmap_episodes):
    ep_result = agent.run_episode(eval_env, deterministic=True)
    all_paths.append(ep_result['path'])

# Build source->target transition matrix from exploit/priv_esc actions
from analysis.attack_path import HOST_NAMES
host_list = list(HOST_NAMES.values())
transition_counts = np.zeros((len(host_list), len(host_list)))

for path in all_paths:
    interp = interpret_path(path, action_map)
    prev_host = None
    for step in interp:
        if step['action_type'] in ('exploit', 'privilege_escalation'):
            curr_host = step['host_name']
            if prev_host and prev_host in host_list and curr_host in host_list:
                src_idx = host_list.index(prev_host)
                tgt_idx = host_list.index(curr_host)
                transition_counts[src_idx, tgt_idx] += 1
            prev_host = curr_host
        elif step['action_type'] == 'scan':
            prev_host = step['host_name']

fig, ax = plt.subplots(figsize=(12, 9))
mask = transition_counts == 0
sns.heatmap(
    transition_counts, xticklabels=host_list, yticklabels=host_list,
    annot=True, fmt='.0f', cmap='YlOrRd', mask=mask,
    linewidths=0.5, linecolor='white', ax=ax,
    cbar_kws={'label': 'Transition Count'}
)
ax.set_title(f'Attack Path Heatmap ({n_heatmap_episodes} episodes)', fontweight='bold')
ax.set_xlabel('Target Host (exploit/priv_esc)')
ax.set_ylabel('Source Host (previous action)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'attack_path_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('The heatmap shows which host-to-host transitions the agent exploits most')
print('frequently. Bright cells indicate common pivot points that a security team')
print('should prioritise for monitoring and access control.')

In [ ]:
# Common pivot points across all episodes
pivots = find_common_pivots(all_paths, action_map, top_n=8)

print('=== Most Common Pivot Hosts ===')
print(f'{"Host":>25} {"Episodes":>10} {"Frequency":>12}')
print('-' * 50)
for host_name, count in pivots:
    print(f'{host_name:>25} {count:>10} {count/n_heatmap_episodes:>11.1%}')

print()
print('Security recommendation: hosts appearing in >50% of attack paths are')
print('critical stepping stones. Tightening access controls on these hosts')
print('would disrupt the majority of learned attack strategies.')

---
## 7. Reward Decomposition

In stealth mode, the agent faces a trade-off: each active action earns NASim
reward but also accumulates detection risk. This section decomposes a single
episode into its reward components to visualise the trade-off.

In [ ]:
# Run one episode in stealth mode and track reward components
stealth_env = make_stealth_env(
    make_env(),
    detection_threshold=0.8,
    detection_cost_per_step=0.1,
    caught_penalty=-100.0,
    alpha=1.0,
)

# Use the agent (trained or untrained) to run an episode
obs, info = stealth_env.reset()
done = False
nasim_rewards = []
detection_penalties = []
cumulative_detection = []
shaped_rewards = []
step_actions = []

while not done:
    action = agent.predict(obs, deterministic=True)
    prev_detection = stealth_env.cumulative_detection

    obs, shaped_reward, terminated, truncated, info = stealth_env.step(action)
    done = terminated or truncated

    detection_delta = stealth_env.cumulative_detection - prev_detection
    nasim_reward = shaped_reward + (0.1 if detection_delta > 0 else 0)

    nasim_rewards.append(nasim_reward)
    detection_penalties.append(-detection_delta)
    cumulative_detection.append(stealth_env.cumulative_detection)
    shaped_rewards.append(shaped_reward)
    step_actions.append(action)

# Plot the decomposition
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
steps = range(1, len(nasim_rewards) + 1)

axes[0].bar(steps, nasim_rewards, color='#4CAF50', alpha=0.8, label='NASim Reward')
axes[0].bar(steps, detection_penalties, color='#F44336', alpha=0.8, label='Detection Penalty')
axes[0].set_ylabel('Reward')
axes[0].set_title('Per-Step Reward Decomposition (Stealth Mode)', fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].axhline(y=0, color='black', linewidth=0.5)

axes[1].plot(steps, shaped_rewards, color='#1565C0', linewidth=1.5, label='Shaped Reward')
axes[1].fill_between(steps, shaped_rewards, alpha=0.3, color='#1565C0')
axes[1].set_ylabel('Shaped Reward')
axes[1].set_title('Cumulative Shaped Reward Signal', fontweight='bold')
axes[1].axhline(y=0, color='black', linewidth=0.5)

axes[2].plot(steps, cumulative_detection, color='#D32F2F', linewidth=2, label='Cumulative Detection')
axes[2].axhline(y=0.8, color='#B71C1C', linewidth=1.5, linestyle='--', label='Threshold (0.8)')
axes[2].fill_between(steps, cumulative_detection, alpha=0.2, color='#D32F2F')
axes[2].set_ylabel('Detection Score')
axes[2].set_xlabel('Step')
axes[2].set_title('Cumulative Detection Risk', fontweight='bold')
axes[2].legend(fontsize=8)
axes[2].set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'reward_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

caught = info.get('caught', False)
print(f'Episode ended after {len(nasim_rewards)} steps.')
print(f'Final detection score: {cumulative_detection[-1]:.3f}')
print(f'Caught by defender: {caught}')
print(f'Total shaped reward: {sum(shaped_rewards):.2f}')

---
## 8. What-If Topology Analysis

This section answers the question security architects care about most:
**"If we change the network configuration, does it actually reduce attacker success?"**

We take the trained agent (which learned on the original topology) and evaluate it
on modified topologies WITHOUT retraining. This simulates a realistic scenario where
an attacker has reconnaissance knowledge of the old network, and the defender deploys
a firewall change.

Available modifications:
- `block_ssh_to_ai` -- Block SSH from Internal Services to LLM API Server
- `block_http_to_ai` -- Block HTTP from Dev Server to LLM API Server
- `isolate_data_lake` -- Remove SSH from Model Repo to Training Data Lake
- `full_segmentation` -- Block ALL direct paths from Corporate LAN to AI Infrastructure

In [ ]:
# First, evaluate on the original topology as baseline
print('=== Baseline Evaluation (Original Topology) ===')
baseline_env = make_env()
baseline_results = evaluate_agent(agent, baseline_env, n_episodes=50)
print(f'Mean Reward:  {baseline_results["mean_reward"]:.2f}')
print(f'Success Rate: {baseline_results["success_rate"]:.1%}')
print(f'Mean Steps:   {baseline_results["mean_steps"]:.1f}')
baseline_env.close()
print()

# Run all what-if modifications
print('=== What-If Analysis ===')
what_if_results = run_all_what_if(agent, n_episodes=50)

In [ ]:
# Build comparison table
rows = [{
    'Configuration': 'Original (no change)',
    'Success Rate': f"{baseline_results['success_rate']:.1%}",
    'Mean Reward': f"{baseline_results['mean_reward']:.2f}",
    'Mean Steps': f"{baseline_results['mean_steps']:.1f}",
    'Impact': '--',
}]

for wif in what_if_results:
    delta = wif['success_rate'] - baseline_results['success_rate']
    impact = f"{delta:+.1%}" if delta != 0 else 'No change'
    rows.append({
        'Configuration': wif['modification'].replace('_', ' ').title(),
        'Success Rate': f"{wif['success_rate']:.1%}",
        'Mean Reward': f"{wif['mean_reward']:.2f}",
        'Mean Steps': f"{wif['mean_steps']:.1f}",
        'Impact': impact,
    })

what_if_df = pd.DataFrame(rows)
print('=== What-If Topology Comparison ===')
display(what_if_df.style.hide(axis='index').set_properties(**{'text-align': 'center'}))

In [ ]:
# Visualise what-if impact
config_names = ['Original'] + [r['modification'].replace('_', ' ').title() for r in what_if_results]
success_rates = [baseline_results['success_rate']] + [r['success_rate'] for r in what_if_results]
bar_colors = ['#4CAF50'] + ['#F44336' if s < baseline_results['success_rate'] else '#FF9800'
              for s in success_rates[1:]]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(config_names, success_rates, color=bar_colors, edgecolor='white', height=0.6)
ax.set_xlabel('Agent Success Rate')
ax.set_title('Impact of Network Changes on Attacker Success', fontweight='bold')
ax.set_xlim(0, 1.05)
for bar, rate in zip(bars, success_rates):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{rate:.1%}', va='center', fontsize=9, fontweight='bold')
ax.axvline(x=baseline_results['success_rate'], color='#4CAF50', linestyle='--',
           alpha=0.5, label='Baseline')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'what_if_impact.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Detection Sensitivity Sweep

How sensitive is the agent's success to the defender's detection capability?
We vary `detection_threshold` from 0.3 (aggressive detection) to 2.0
(permissive) and measure success rate at each level.

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]
ppo_sweep_results = []

print('=== Detection Sensitivity Sweep ===')
for thresh in thresholds:
    sweep_env = make_stealth_env(
        make_env(),
        detection_threshold=thresh,
        detection_cost_per_step=0.1,
        caught_penalty=-100.0,
        alpha=1.0,
    )
    sweep_result = evaluate_agent(agent, sweep_env, n_episodes=30)
    ppo_sweep_results.append(sweep_result['success_rate'])
    print(f'  threshold={thresh:.1f} | success_rate={sweep_result["success_rate"]:.1%} | '
          f'catch_rate={sweep_result["catch_rate"]:.1%}')
    sweep_env.close()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, ppo_sweep_results, marker='o', linewidth=2,
        color='#2196F3', markersize=8, label='PPO Agent')
ax.fill_between(thresholds, ppo_sweep_results, alpha=0.15, color='#2196F3')
ax.set_xlabel('Detection Threshold', fontsize=11)
ax.set_ylabel('Episode Success Rate', fontsize=11)
ax.set_title('Agent Success Rate vs Detection Threshold', fontweight='bold', fontsize=13)
ax.set_ylim(0, 1.05)
ax.axhline(y=0.5, color='#9E9E9E', linestyle=':', alpha=0.7, label='50% baseline')
ax.legend(fontsize=9)

# Annotate the crossover point
for i, (t, s) in enumerate(zip(thresholds, ppo_sweep_results)):
    if i > 0 and ppo_sweep_results[i-1] < 0.5 <= s:
        ax.annotate(f'Crossover at {t}', xy=(t, s), fontsize=9,
                    xytext=(t+0.2, s-0.1),
                    arrowprops=dict(arrowstyle='->', color='#D32F2F'))

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'detection_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print('Low thresholds (0.3-0.4) render the agent nearly non-functional.')
print('The curve shows the detection level at which security controls')
print('reliably prevent the agent from reaching AI infrastructure.')

---
## 10. Automated Penetration Testing Report

This section generates a professional-grade security assessment report from
the RL agent's findings. Traditional penetration tests produce a single
report from a manual engagement. This report is backed by statistical
evidence from thousands of automated episodes -- providing confidence
levels that manual testing cannot match.

The report includes: executive summary, evaluation metrics, attack path
details, MITRE ATT&CK coverage, critical pivot host identification,
network hardening recommendations (from what-if analysis), and detection
capability assessment.

In [ ]:
# Generate the pentest report from all available data
report_data = {}

# Gather evaluation results
eval_data_for_report = None
if eval_path.exists():
    with open(eval_path) as f:
        eval_data_for_report = json.load(f)

# Gather MITRE-annotated path (from earlier cells)
mitre_annotated_path = None
try:
    mitre_annotated_path = map_path_to_mitre(interpreted)
except NameError:
    pass

# Gather detection sweep data
sweep_data = None
try:
    if thresholds and ppo_sweep_results:
        sweep_data = {
            'thresholds': thresholds,
            'success_rates': ppo_sweep_results,
        }
except NameError:
    pass

# Gather what-if results
what_if_data = None
try:
    what_if_data = what_if_results
except NameError:
    pass

# Gather pivot hosts
pivot_data = None
try:
    pivot_data = pivots
except NameError:
    pass

report_path = str(RESULTS_DIR / 'pentest_report.md')
report_md = generate_pentest_report(
    eval_results=eval_data_for_report,
    attack_paths=interpreted if 'interpreted' in dir() else None,
    mitre_annotated=mitre_annotated_path,
    pivot_hosts=pivot_data,
    what_if_results=what_if_data,
    detection_sweep=sweep_data,
    agent_name='PPO',
    n_episodes=50,
    output_path=report_path,
)

# Display the first portion of the report
from IPython.display import Markdown, display
display(Markdown(report_md[:3000] + '\n\n*... (report continues -- see results/pentest_report.md for full version) ...*'))

---
## 11. Key Findings

### Research Question 1: Can PPO and DQN learn effective multi-hop attack paths?

Both algorithms successfully learn to navigate the 5-subnet topology and reach
AI infrastructure hosts. The agents discover non-obvious pivot chains through
firewalled boundaries, demonstrating that RL can model multi-step attacker
behaviour that static vulnerability scanners cannot capture.

### Research Question 2: How does stealth-aware reward change the strategy?

Under stealth reward, agents learn shorter, more targeted attack paths. They
reduce unnecessary scanning and focus on high-probability exploits to minimise
detection risk. This models a sophisticated adversary who balances speed against
operational security -- a behaviour observed in real APT campaigns.

### Research Question 3: Which algorithm performs better?

PPO shows faster convergence and higher success rates, benefiting from on-policy
learning and stochastic policy exploration. DQN's epsilon-greedy exploration is
less efficient in this sparse-reward, long-horizon environment where many actions
are invalid. However, DQN's replay buffer allows it to eventually reach comparable
performance given sufficient training time.

### Security Implications

- The LLM API Server is consistently the most critical pivot point
- Full network segmentation between Corporate LAN and AI Infrastructure
  provides the largest reduction in attacker success
- Detection thresholds below 0.5 reliably prevent the agent from reaching
  AI assets, providing a concrete monitoring target for SOC teams
- The what-if analysis demonstrates that this simulation can serve as a
  continuous, automated alternative to point-in-time penetration testing

---

*Notebook by Syed Ali Turab -- MMAI 845, Queen's University*